# pricing-the-match, step 1: from past results to match probabilities

**Question:** can past football results be turned into sensible probabilities for home win, draw and away win, and how do they compare with a bookmaker's?

This is the first of four steps:

| Step | Question |
|---|---|
| **1 (this notebook)** | Can past results be turned into match probabilities? |
| 2 | How do probabilities become odds with a margin? |
| 3 | What does the operator's profit depend on? |
| 4 | Web app and build log |

Run the cells top to bottom. Needs `pandas`, `numpy`, `scipy`, `scikit-learn`, `matplotlib`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import poisson
from sklearn.linear_model import LogisticRegression

REQUIRED = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]
ODDS = ["B365H", "B365D", "B365A"]      # Bet365 home / draw / away decimal odds
CLASSES = ["H", "D", "A"]

## 1. Data

**Real data:** season files from football-data.co.uk (Premier League is `E0.csv`). Put one CSV per season in the project's `data/real` folder, renamed so they sort in time order (`E0_2019.csv`, `E0_2020.csv`, ...). The last file by name becomes the test season.

**Synthetic data:** if `data/real` is empty, the notebook generates fake seasons in the same format so everything still runs. In the fake league the bookmaker knows the true probabilities, so it should beat any model. Synthetic numbers show the code works; they are **not** findings about football.

**Where the data is looked for.** All paths are relative to the folder the notebook runs in: first `data/real`, then `../data/real` (the project's own data folder when the notebook is opened from `notebooks/`). If your files are somewhere else, add that folder to `CANDIDATES` in the next cell, also as a relative path. In Google Colab either upload the CSVs to a `data/real` folder in the session's file panel, or mount Google Drive (`from google.colab import drive; drive.mount("drive")`) and add `Path("drive/MyDrive/<your folder>")`.

In [ ]:
def load_matches(folder):
    """Read every CSV in `folder` (one per season) into one table sorted by date."""
    files = sorted(Path(folder).glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files in {folder}")
    frames = []
    for f in files:
        df = pd.read_csv(f, encoding="latin-1")
        missing = [c for c in REQUIRED + ODDS if c not in df.columns]
        if missing:
            raise ValueError(f"{f.name} is missing columns {missing}")
        df = df[REQUIRED + ODDS].dropna(subset=REQUIRED)
        # the source mixes dd/mm/yy and dd/mm/yyyy across seasons
        df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, format="mixed")
        df["season"] = f.stem
        frames.append(df)
    out = pd.concat(frames, ignore_index=True).sort_values("Date", kind="stable")
    out = out.dropna(subset=ODDS).reset_index(drop=True)
    bad = ~out["FTR"].isin(CLASSES)
    if bad.any():
        raise ValueError(f"{bad.sum()} rows have a result other than H/D/A")
    return out


def make_synthetic(folder, seasons=6, teams=20, margin=0.05, seed=7):
    """Write fake seasons in football-data.co.uk format.

    Each team has a hidden strength that drifts between seasons; goals are
    Poisson. The fake bookmaker knows the true probabilities up to a little
    noise and adds `margin` on top.
    """
    rng = np.random.default_rng(seed)
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    names = [f"Team{i:02d}" for i in range(teams)]
    strength = rng.normal(0, 0.35, teams)
    goals = np.arange(0, 11)
    for s in range(seasons):
        rows = []
        day = pd.Timestamp(2019 + s, 8, 10)
        fixtures = [(h, a) for h in range(teams) for a in range(teams) if h != a]
        rng.shuffle(fixtures)
        for i, (h, a) in enumerate(fixtures):
            lam_h = np.exp(0.30 + strength[h] - strength[a])   # 0.30 = home edge
            lam_a = np.exp(0.05 + strength[a] - strength[h])
            grid = np.outer(poisson.pmf(goals, lam_h), poisson.pmf(goals, lam_a))
            p = np.array([np.tril(grid, -1).sum(), np.trace(grid), np.triu(grid, 1).sum()])
            p = p / p.sum()
            gh, ga = rng.poisson(lam_h), rng.poisson(lam_a)
            noisy = p * np.exp(rng.normal(0, 0.03, 3))
            noisy = noisy / noisy.sum()
            odds = np.round(1 / (noisy * (1 + margin)), 2)
            rows.append({
                "Date": (day + pd.Timedelta(days=i // 10 * 7)).strftime("%d/%m/%Y"),
                "HomeTeam": names[h], "AwayTeam": names[a],
                "FTHG": gh, "FTAG": ga,
                "FTR": "H" if gh > ga else ("D" if gh == ga else "A"),
                "B365H": odds[0], "B365D": odds[1], "B365A": odds[2],
            })
        pd.DataFrame(rows).to_csv(folder / f"synthetic_{2019 + s}.csv", index=False)
        strength = 0.85 * strength + rng.normal(0, 0.12, teams)

In [ ]:
CANDIDATES = [Path("data/real"), Path("../data/real")]   # relative to where the notebook runs; add your own folder here
REAL = next((c for c in CANDIDATES if list(c.glob("*.csv"))), CANDIDATES[0])
SYNTH = Path("data/synthetic")

if list(REAL.glob("*.csv")):
    DATA, IS_REAL = REAL, True
    print(f"Using REAL season files from {REAL}")
else:
    DATA, IS_REAL = SYNTH, False
    if not list(SYNTH.glob("*.csv")):
        make_synthetic(SYNTH)
    print("No real CSVs found, so using SYNTHETIC seasons (not real football)")

raw = load_matches(DATA)
print(f"{len(raw)} matches across {raw['season'].nunique()} seasons")
raw.head()

## 2. Elo ratings

Every team starts at 1500. After each match the winner takes points from the loser, and the size of the transfer depends on how surprising the result was:

$$E_{home} = \frac{1}{1 + 10^{-(R_{home} + \text{home advantage} - R_{away})/400}}, \qquad \Delta = K\,(S_{home} - E_{home})$$

where $S_{home}$ is 1 for a home win, 0.5 for a draw, 0 for an away win.

**The important detail:** the rating attached to a match is the rating *before* that match. Using the post-match rating would leak the result into the feature. That look-ahead bug is the most common mistake in this kind of model, and section 5 tests for it.

In [ ]:
START = 1500.0

def add_elo(matches, k=20.0, home_adv=60.0):
    """Return a copy of `matches` (must be date-sorted) with pre-match
    elo_home, elo_away and elo_diff (home minus away)."""
    if not matches["Date"].is_monotonic_increasing:
        raise ValueError("matches must be sorted by date")
    rating = {}
    eh, ea = [], []
    for home, away, res in zip(matches["HomeTeam"], matches["AwayTeam"], matches["FTR"]):
        rh, ra = rating.get(home, START), rating.get(away, START)
        eh.append(rh)                      # stored BEFORE the update
        ea.append(ra)
        expected_home = 1.0 / (1.0 + 10 ** (-(rh + home_adv - ra) / 400.0))
        score_home = {"H": 1.0, "D": 0.5, "A": 0.0}[res]
        delta = k * (score_home - expected_home)
        rating[home], rating[away] = rh + delta, ra - delta
    out = matches.copy()
    out["elo_home"], out["elo_away"] = eh, ea
    out["elo_diff"] = out["elo_home"] - out["elo_away"]
    out.attrs["final_ratings"] = rating
    return out

matches = add_elo(raw)
final = pd.Series(matches.attrs["final_ratings"]).sort_values(ascending=False)
print("Strongest and weakest teams by final rating:")
pd.concat([final.head(5), final.tail(5)]).round(0).to_frame("elo")

## 3. The bookmaker benchmark

Decimal odds of 2.00 imply a probability of 1/2.00 = 50%. For one match the three implied probabilities add up to **more than 100%**. The excess is the bookmaker's margin, called the *overround*. Dividing by the total removes it and leaves the bookmaker's own probability estimate.

This is the first glimpse of "how the house makes money", which step 2 builds on.

In [ ]:
def implied_probs(odds):
    """odds (n,3) -> (fair probabilities (n,3), overround (n,))."""
    raw_p = 1.0 / np.asarray(odds, dtype=float)
    total = raw_p.sum(axis=1)
    return raw_p / total[:, None], total - 1.0

ex = matches.iloc[[len(matches) // 2]]
p_ex, over_ex = implied_probs(ex[ODDS].to_numpy())
print(f"{ex['HomeTeam'].iloc[0]} v {ex['AwayTeam'].iloc[0]}")
print("odds          ", ex[ODDS].to_numpy()[0])
print("raw 1/odds    ", (1 / ex[ODDS].to_numpy()[0]).round(3), "-> sum", (1 / ex[ODDS].to_numpy()[0]).sum().round(3))
print("fair probs    ", p_ex[0].round(3))
print(f"overround      {over_ex[0]:.2%}")

## 4. The model

A multinomial logistic regression with **one feature**: the pre-match rating gap. Deliberately simple, so the result is three curves that can be plotted and explained.

The split is by season: train on the past, test on the most recent season. A random split would let the model train on matches played *after* the ones it is tested on.

In [ ]:
def split_by_season(df, test_seasons=1):
    seasons = list(dict.fromkeys(df["season"]))
    if len(seasons) <= test_seasons:
        raise ValueError("need more seasons than test_seasons")
    is_test = df["season"].isin(seasons[-test_seasons:])
    return df[~is_test], df[is_test]

def fit(train):
    model = LogisticRegression(C=1e6, max_iter=1000)   # large C = no regularisation
    model.fit(train[["elo_diff"]].to_numpy() / 100.0, train["FTR"])
    return model

def predict(model, df):
    """(n,3) probabilities in H, D, A order."""
    p = model.predict_proba(df[["elo_diff"]].to_numpy() / 100.0)
    order = [list(model.classes_).index(c) for c in CLASSES]
    return p[:, order]

train, test = split_by_season(matches)
model = fit(train)
print(f"train {len(train)} matches | test {len(test)} matches (season {test['season'].iloc[0]})")

grid = pd.DataFrame({"elo_diff": np.linspace(-300, 300, 121)})
pg = predict(model, grid)
fig, ax = plt.subplots(figsize=(7, 4))
for j, name in enumerate(["home win", "draw", "away win"]):
    ax.plot(grid["elo_diff"], pg[:, j], label=name)
ax.set_xlabel("Elo gap (home minus away)")
ax.set_ylabel("probability")
ax.set_title("What the model says for a given rating gap")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 5. How good is it?

Three forecasters are scored on the test season:

- **base rates**: always predicts the historical share of home wins, draws, away wins
- **elo + logistic**: the model above
- **bookmaker**: implied probabilities with the margin removed

Two scores, both *lower is better*:

- **Brier score**: mean squared error of the probabilities. Guessing 1/3 each scores 0.667.
- **Log loss**: punishes confident wrong answers harder. Guessing 1/3 each scores 1.099.

In [ ]:
def one_hot(results):
    return np.stack([(results == c).to_numpy(dtype=float) for c in CLASSES], axis=1)

def brier(p, y):
    return float(((p - y) ** 2).sum(axis=1).mean())

def log_loss(p, y):
    return float(-(y * np.log(np.clip(p, 1e-12, 1))).sum(axis=1).mean())

def evaluate(train, test, model):
    y = one_hot(test["FTR"])
    base = np.tile(one_hot(train["FTR"]).mean(axis=0), (len(test), 1))
    ours = predict(model, test)
    book, _ = implied_probs(test[ODDS].to_numpy())
    rows = [(name, brier(p, y), log_loss(p, y))
            for name, p in [("base rates", base), ("elo + logistic", ours), ("bookmaker", book)]]
    return pd.DataFrame(rows, columns=["forecaster", "brier", "log_loss"]).round(4)

_, over = implied_probs(test[ODDS].to_numpy())
print("REAL data" if IS_REAL else "SYNTHETIC data: shows the code works, not a finding")
print(f"Average bookmaker overround on the test season: {over.mean():.2%}")
evaluate(train, test, model)

**Calibration.** When the model says 40%, does it happen about 40% of the time? All three outcomes are pooled, bucketed by predicted probability, and compared with how often they occurred. Points on the diagonal mean well calibrated.

In [ ]:
def calibration(p, y, bins=5):
    flat_p, flat_y = p.ravel(), y.ravel()
    edges = np.quantile(flat_p, np.linspace(0, 1, bins + 1))
    idx = np.clip(np.searchsorted(edges, flat_p, side="right") - 1, 0, bins - 1)
    t = pd.DataFrame({"bin": idx, "predicted": flat_p, "observed": flat_y})
    return t.groupby("bin").agg(predicted=("predicted", "mean"),
                                observed=("observed", "mean"),
                                n=("observed", "size")).round(3)

y_test = one_hot(test["FTR"])
cal_ours = calibration(predict(model, test), y_test)
cal_book = calibration(implied_probs(test[ODDS].to_numpy())[0], y_test)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
ax.plot(cal_ours["predicted"], cal_ours["observed"], "o-", label="elo + logistic")
ax.plot(cal_book["predicted"], cal_book["observed"], "s-", label="bookmaker")
ax.set_xlabel("predicted probability"); ax.set_ylabel("observed frequency")
ax.set_xlim(0, 0.9); ax.set_ylim(0, 0.9)
ax.set_title("Calibration on the test season")
ax.legend(); ax.grid(alpha=0.3)
plt.show()
cal_ours

## 6. Checks

These run on a small synthetic league so they work whatever is in `data/real`. Each one guards against a specific mistake. If every line prints `ok`, the pipeline is sound.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    make_synthetic(tmp, seasons=4, teams=12)
    chk = add_elo(load_matches(tmp))

# loader
assert chk["Date"].is_monotonic_increasing and len(chk) == 4 * 12 * 11
print("ok  loader returns every match, sorted by date")

# Elo starts at 1500 and only moves points between teams
assert chk.loc[0, "elo_home"] == 1500 and chk.loc[0, "elo_away"] == 1500
assert np.isclose(np.mean(list(chk.attrs["final_ratings"].values())), 1500)
print("ok  Elo starts at 1500 and the league average never moves (zero-sum)")

# NO LOOK-AHEAD: flipping the last result must not change any pre-match rating
flipped = chk[["Date", "HomeTeam", "AwayTeam", "FTR"]].copy()
flipped.loc[flipped.index[-1], "FTR"] = "A" if flipped["FTR"].iloc[-1] != "A" else "H"
assert np.allclose(add_elo(flipped)["elo_diff"], chk["elo_diff"])
print("ok  no look-ahead: a match's rating does not depend on its own result")

# unsorted input is refused rather than silently producing wrong ratings
try:
    add_elo(chk.iloc[::-1]); raise AssertionError("unsorted input accepted")
except ValueError:
    print("ok  unsorted input is rejected")

# implied probabilities
p_, o_ = implied_probs(np.array([[2.0, 4.0, 4.0], [1.9, 3.5, 4.2]]))
assert np.allclose(p_.sum(axis=1), 1) and np.isclose(o_[0], 0) and o_[1] > 0
print("ok  implied probabilities sum to 1; a fair book has zero overround")

# model output
tr_, te_ = split_by_season(chk)
m_ = fit(tr_)
pp = predict(m_, te_)
assert np.allclose(pp.sum(axis=1), 1) and (pp >= 0).all()
home = predict(m_, pd.DataFrame({"elo_diff": [-200.0, 0.0, 200.0]}))[:, 0]
assert home[0] < home[1] < home[2]
assert tr_["Date"].max() < te_["Date"].min()
print("ok  predictions are probabilities, rise with the rating gap, and the split is chronological")

# scores against values worked out by hand
y1 = np.array([[1.0, 0, 0]])
assert brier(y1, y1) == 0 and np.isclose(brier(np.full((1, 3), 1 / 3), y1), 2 / 3)
assert np.isclose(log_loss(np.full((1, 3), 1 / 3), y1), np.log(3))
print("ok  Brier and log loss match hand-calculated values")

# known ordering on synthetic data
ll = evaluate(tr_, te_, m_).set_index("forecaster")["log_loss"]
assert ll["bookmaker"] < ll["elo + logistic"] < ll["base rates"]
print("ok  on synthetic data: bookmaker < model < base rates, as designed")

## 7. What this shows, and what comes next

- A single number per team, updated match by match, already beats the base rates.
- The bookmaker is the benchmark to respect. On real data, expect it to stay ahead; the size of the gap is the honest headline.
- The bookmaker's prices carry a built-in margin (the overround). That margin is where an operator's profit starts.

**Step 2** takes these probabilities and turns them into odds with a chosen margin. **Step 3** simulates a book of bettors to see how profit and risk depend on margin, bet limits and model error.